# 1. working_fixed: assign groups

### [read] Summarise data availability of addresses among the fixed_table
- So we know what we're working with in terms of different types

In [2]:
# ### [read] Summarise data availability of addresses among the fixed_table
# - So we know what we're working with in terms of different types
# This should be lat/long, full address, address decomposed across the different fields, just postcode, the nothing
# And if its primary trading address or registered office address
import ibis

address_hierachy = [
    {'primary_trading_address_latitude', 'primary_trading_address_longitude'},
    {'primary_trading_address'},
    {'ro_latitude', 'ro_longitude'},
    {'ro_address'},
    {'ro_address_line_1', 'ro_full_postcode'},
    {'ro_full_postcode'},
    {'ro_postcode'}
]

# Dynamically builds an ibis.cases() expression from a list of column sets.
def build_location_source_cases(table: ibis.expr.types.Table, hierarchy: list[set]):
    cases_list = []
    
    for i, field_set in enumerate(hierarchy, start=1):
        # 1. Build the logical condition: EVERY column in the set must be not-null
        condition = None
        for col_name in field_set:
            is_not_null = table[col_name].notnull()
            condition = is_not_null if condition is None else condition & is_not_null
        
        # 2. Store as a (condition, result_value) tuple
        cases_list.append((condition, i))
        
    # 3. Unpack the list of tuples into ibis.cases. 
    # Fallback MUST be an integer to match the column type.
    fallback_lvl = len(hierarchy) + 1
    return ibis.cases(*cases_list, else_=fallback_lvl)

### [write] Add the ONS postcode directory to the database

In [2]:
make_xlsx = False
make_short = False
make_skinny = False
make_split = False

import pandas as pd
import numpy as np
from numpy.random import rand
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="calculations")
table_name = "ref_ons_postcode"
file_name = "skinny_NSPL_MAY_2026_UK.csv"

if make_xlsx:
    csv_file = f"{dirs.input_dir}/{file_name}"
    df_raw = pd.read_csv(csv_file)
    df_raw.to_excel(f"{dirs.output_dir}/{file_name.replace('.csv', '.xlsx')}", index=False)

if make_split:
    splits = 2
    csv_file = f"{dirs.input_dir}/{file_name}"
    
    # Still loads entirely into memory
    df_raw = pd.read_csv(csv_file) 
    
    # Automatically handles the math to split into relatively equal chunks
    nps_split = np.array_split(df_raw, splits)
    
    assert len(df_raw) == sum(len(df) for df in nps_split), "Split failed: row counts do not match."
    
    for i, np_split in enumerate(nps_split):
        df_split = pd.DataFrame(np_split, columns=df_raw.columns)
        df_split.to_csv(f"{dirs.input_dir}/split_{i+1}_{file_name}", index=False)


# Skip 99.9% of the rows
if make_short:
    csv_file = f"/mnt/c/Users/lazym/OneDrive - University College London/Diss 2/Data/Postcode/{file_name}"
    df_raw = pd.read_csv(csv_file, skiprows=lambda x: x > 0 and rand() > 0.001)
    df_raw_short = df_raw.sample(500)
    df_raw_short.to_csv(f"{dirs.input_dir}/short_{file_name}", index=False)

In [4]:
import ibis
from ibis import _

table_name = "ref_ons_postcode"
file_name = "skinny_NSPL_MAY_2026_UK.csv"

# Initialize connection
con = ibis.duckdb.connect(str(dirs.db_path))
keep_cols = { "pcds", "lat", "long", "ttwa15cd" }

table_csv = con.read_csv(f"{dirs.input_dir}/split_*_{file_name}")

selected_cols = list(keep_cols.intersection(set(table_csv.columns)))

table_ref = table_csv.select(selected_cols)
rows_count = table_ref.count().execute()
print(f"{rows_count:,} rows in new table {table_name}")
print("Sample:")
display(table_ref.sample(10 / rows_count).execute())
table_out = con.create_table(table_name, table_ref, overwrite=table_name in con.list_tables())

2,726,477 rows in new table ref_ons_postcode
Sample:


,ttwa15cd,lat,long,pcds
0,E30000029,53.718642,-1.857525,HX1 2UE
1,E30000264,53.286823,-0.065895,LN11 9XU
2,E30000237,51.887484,-0.441093,LU4 8AX
3,E30000245,55.001014,-1.590972,NE7 7AT
4,E30000247,52.240565,-0.875383,NN1 5ZT
5,E30000248,52.603235,1.230971,NR4 6FX
6,E30000234,51.527370,-0.165718,NW8 7JN
7,E30000255,53.701536,-2.660480,PR25 5TA
8,E30000004,53.521957,-1.619842,S36 6GB
9,E30000234,51.493501,0.065350,SE18 6DH


In [6]:
import ibis
from ibis import _

table_name = "ref_ons_ttwa_name"
file_name = "TTWA Travel to Work Area names and codes UK as at 12_11 v5.csv"

# Initialize connection
con = ibis.duckdb.connect(str(dirs.db_path))
table_csv = con.read_csv(f"{dirs.input_dir}/{file_name}")

table_ttwa_ref = table_csv
rows_count = table_ttwa_ref.count().execute()
print(f"{rows_count:,} rows in new table {table_name}")
print("Sample:")
display(table_ttwa_ref.sample(10 / rows_count).execute())
table_out = con.create_table(table_name, table_ttwa_ref, overwrite=table_name in con.list_tables())

230 rows in new table ref_ons_ttwa_name
Sample:


,TTWA11CD,TTWA11NM
0,E30000018,Bradford
1,E30000108,Peterborough
2,E30000173,Blyth and Ashington
3,E30000175,Bournemouth
4,E30000200,Derby
5,E30000203,Durham and Bishop Auckland
6,S22000053,Dalbeattie and Castle Douglas
7,S22000062,Fort William


### [write] Create new column for address geocoding
- The third decimal place is worth up to 110 m: it can identify a large agricultural field or institutional campus.
- The fourth decimal place is worth up to 11 m: it can identify a parcel of land. It is comparable to the typical accuracy of an uncorrected GPS unit with no interference.
- The fifth decimal place is worth up to 1.1 m: it distinguish trees from each other. Accuracy to this level with commercial GPS units can only be achieved with differential correction.

In [ ]:
import ibis
from ibis import _, selectors
import pandas as pd
from utils.f_0_dirs import get_data_dirs
from f_4_spatial import convert_dms_to_decimal

old_table_name = "fame_fixed_filtered"
new_table_name = "working_fixed"

# Initialize connection
dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))
con.raw_sql("INSTALL spatial; LOAD spatial;")

# Reference the existing tables
fame_fixed = con.table(old_table_name)

# 1. Parse availability into an indicator and extract the best available text address
sep: ibis.StringScalar = ibis.literal(", ", type="string")
working_raw = fame_fixed.mutate(
    address_raw_lvl = build_location_source_cases(fame_fixed, address_hierachy),
    address_raw = ibis.coalesce(
        fame_fixed.primary_trading_address,
        fame_fixed.ro_address,
        # Fallback: concatenate the separate lines and postcode if above are null
        sep.join(
            ibis.array([
                fame_fixed.ro_address_line_1, 
                fame_fixed.ro_address_line_2, 
                fame_fixed.ro_full_postcode
            ]).filter(lambda x: x.notnull())
        ),
        fame_fixed.ro_full_postcode,
        fame_fixed.ro_postcode
    )
)

# 2. Extract and clean postcodes for Levels 2, 4, 5, 6, and 7
# Regex matches standard UK postcode formats. We strip whitespace and uppercase for perfect joins.
# 2. Extract and clean postcodes for Levels 2, 4, 5, 6, and 7
# Regex matches standard UK postcode formats. 
uk_postcode_regex = r"([A-Za-z]{1,2}\d[A-Za-z\d]?\s?\d[A-Za-z]{2})"

working_pc = working_raw.mutate(
    extracted_postcode = ibis.cases(
        (working_raw.address_raw_lvl.isin([1, 2]), working_raw.address_raw.re_extract(uk_postcode_regex, 1)),
        (working_raw.address_raw_lvl.isin([3, 4]), working_raw.ro_address.re_extract(uk_postcode_regex, 1)),
        (working_raw.address_raw_lvl.isin([5, 6]), working_raw.ro_full_postcode),
        (working_raw.address_raw_lvl == 7, working_raw.ro_postcode),
        else_=ibis.literal(None, type="string")
    )
    .cast("string")
    .upper()
    .re_replace(r"\s+", "") # Step 1: Strip all existing spaces (e.g. "EC1Y2AL" or "EC1Y  2AL" -> "EC1Y2AL")
    .re_replace(r"(.+)(\d[A-Z]{2})$", r"\1 \2") # Step 2: Insert exactly one space before the 3-character inward code
)

# Clean the ONS directory postcodes using the same logic for the join
ons_table_name = "ref_ons_postcode" # Assume this exists with 'postcode', 'lat', 'long'
ons_lookup = con.table(ons_table_name) 

# 3. Join firm data with the ONS lookup
working_joined = working_pc.left_join(
    ons_lookup,
    working_pc.extracted_postcode == ons_lookup.pcds
)

# 4. Extract standardized spatial coordinates based on ALL 7 levels of the hierarchy
working_fixed = (
    working_joined
    .mutate(
        address_case = ibis.cases(
            (working_joined.address_raw_lvl <= 2, ibis.literal('pta')),
            (working_joined.address_raw_lvl <= 7, ibis.literal('ro')),
            else_=ibis.literal(None, type="string")
        ),
        address_lvl = ibis.cases(
            (working_joined.address_raw_lvl <= 2, working_joined.address_raw_lvl),
            (working_joined.address_raw_lvl <= 7, working_joined.address_raw_lvl - 2),
            else_=ibis.literal(None, type="int64")
        ),
        lat_dec = ibis.cases(
            (working_joined.address_raw_lvl == 1, convert_dms_to_decimal(working_joined.primary_trading_address_latitude)),
            (working_joined.address_raw_lvl == 3, convert_dms_to_decimal(working_joined.ro_latitude)),
            else_=working_joined.lat # ibis.literal(None, type='float64') # # Automatically covers levels 2, 4, 5, 6, and 7
        ),
        lon_dec = ibis.cases(
            (working_joined.address_raw_lvl == 1, convert_dms_to_decimal(working_joined.primary_trading_address_longitude)),
            (working_joined.address_raw_lvl == 3, convert_dms_to_decimal(working_joined.ro_longitude)),
            else_=working_joined.long # ibis.literal(None, type='float64') # # Automatically covers levels 2, 4, 5, 6, and 7
        ),
        ttwa = working_joined.ttwa15cd,
        pc4 = working_joined.extracted_postcode.split(" ")[0],
    )
    .mutate(
        lat_lon5 = ibis._.lat_dec.round(5).cast("string") + "," + ibis._.lon_dec.round(5).cast("string")
    )
)

working_fixed_loc = (
    working_fixed
    .select('registered_number', 'address_raw', 'extracted_postcode', 'address_case', 'address_lvl', 'lat_dec', 'lon_dec', 'ttwa', 'pc4', 'lat_lon5')
    # .drop("extracted_postcode", "postcode", "lat", "long") # Drop join artifacts
    .mutate(
        # We need to split by " ", not just take the first 4 characters, because some postcodes have a space in the middle (e.g. "EC1A 1BB" -> "EC1A")
        lat_lon4 = working_fixed.lat_dec.round(4).cast("string") + "," + working_fixed.lon_dec.round(4).cast("string"),
        lat_lon3 = working_fixed.lat_dec.round(3).cast("string") + "," + working_fixed.lon_dec.round(3).cast("string")
    )
)

row_count = working_fixed_loc.count().execute()
display(working_fixed_loc.sample(10 / row_count).execute())

# Filter for those which extracted_postcode is NaN
# Count and display those rows (max 50)
nan_rows = working_fixed_loc.filter(working_fixed_loc.extracted_postcode.isnull())
nan_count = nan_rows.count().execute()
print(f"Number of rows with NaN extracted_postcode: {nan_count}={nan_count/row_count*100:.1f}%")
if nan_count > 0:
    display(nan_rows.sample(50 / nan_count).execute())
    for row in nan_rows.limit(50).execute().to_dict(orient='records'):
        print(row)

# Summary stats: display as a dataframe
# columns should be should be: unique counts, percentage of NaN values,average number of peers per group (i.e. rows / unique counts)
# rows should be for each property: ttwa, first half of extracted_postcode, full extracted_postcode, and latitude and longitude (combined and rounded to 5 decimal places)
stats_g_properties = ['ttwa', 'pc4', 'extracted_postcode', 'lat_lon5', 'lat_lon4', 'lat_lon3']
stats_df = pd.DataFrame(columns=['property', 'unique_count', 'null_count', 'nan_percentage', 'avg_peers_per_group'])
stats_table = working_fixed_loc.select(stats_g_properties)
for prop in stats_g_properties:
    unique_count = stats_table[prop].nunique().execute()
    avg_peers_per_group = row_count / unique_count if unique_count > 0 else np.nan
    null_count = stats_table[prop].isnull().sum().execute()
    nan_percentage = null_count / row_count * 100
    stats_df = pd.concat([stats_df, pd.DataFrame({
        'property': [prop],
        'unique_count': [unique_count],
        'avg_peers_per_group': [avg_peers_per_group],
        'null_count': [null_count],
        'nan_percentage': [nan_percentage]
    })] , ignore_index=True)
display(
    stats_df.style
    .format({
        'unique_count': '{:,.0f}',
        'null_count': '{:,.0f}',
        'nan_percentage': '{:.1f}%',
        'avg_peers_per_group': '{:,.2f}'
    })
)

# Display rows where ttwa is null.
print("Travel to Work Area (TTWA) is null:")
ttwa_null_rows = working_fixed_loc.filter(working_fixed_loc.ttwa.isnull()).execute()
display(ttwa_null_rows)

# Display rows where lat_lon5 is null.
print("Latitude and Longitude (rounded to 5 decimal places) is null:")
lat_lon5_null_rows = working_fixed_loc.filter(working_fixed_loc.lat_lon5.isnull()).execute()
display(lat_lon5_null_rows)

,registered_number,address_raw,extracted_postcode,address_case,address_lvl,lat_dec,lon_dec,ttwa,pc4,lat_lon5,lat_lon4,lat_lon3
0,04722006,"The Barracks 400 Bolton Road, Bury, Lancashire...",BL8 2DA,pta,2,53.587143,-2.323322,E30000239,BL8,"53.58714,-2.32332","53.5871,-2.3233","53.587,-2.323"
1,07471527,"10 Queen Street Place, London, EC4R 1AG",EC4R 1AG,ro,1,51.510667,-0.093139,E30000234,EC4R,"51.51067,-0.09314","51.5107,-0.0931","51.511,-0.093"
2,04469280,"Pickford House, 20 High View Close, Hamilton, ...",LE4 9LJ,ro,2,52.656986,-1.084072,E30000230,LE4,"52.65699,-1.08407","52.657,-1.0841","52.657,-1.084"
3,01078691,"Crownthorpe, Wicklewood, Wymondham, Norfolk, N...",NR18 9EP,pta,1,52.584444,1.080500,E30000248,NR18,"52.58444,1.0805","52.5844,1.0805","52.584,1.081"
4,05933013,"Unit 14-15, Evolution, Whittle Way, Catcliffe,...",S60 5BL,pta,2,53.384753,-1.378782,E30000261,S60,"53.38475,-1.37878","53.3848,-1.3788","53.385,-1.379"
5,10589826,"New Kings Court, Tollgate, Chandler's Ford, Ea...",SO53 3LG,pta,2,50.965373,-1.384775,E30000267,SO53,"50.96537,-1.38478","50.9654,-1.3848","50.965,-1.385"
6,02681512,"St. Catherine's School Twickenham, Cross Deep,...",TW1 4QJ,pta,2,51.443207,-0.330621,E30000266,TW1,"51.44321,-0.33062","51.4432,-0.3306","51.443,-0.331"
7,07025891,"13 Lowthian Road, Hartlepool, Cleveland, TS24 8BH",TS24 8BH,pta,1,54.687833,-1.216611,E30000215,TS24,"54.68783,-1.21661","54.6878,-1.2166","54.688,-1.217"


Number of rows with NaN extracted_postcode: 0=0.0%


,property,unique_count,null_count,nan_percentage,avg_peers_per_group
0,ttwa,228,567,0.4%,668.33
1,pc4,"2,705",0,0.0%,56.33
2,extracted_postcode,"49,963",0,0.0%,3.05
3,lat_lon5,"53,246",507,0.3%,2.86
4,lat_lon4,"53,193",507,0.3%,2.86
5,lat_lon3,"48,540",507,0.3%,3.14


Travel to Work Area (TTWA) is null:


,registered_number,address_raw,extracted_postcode,address_case,address_lvl,lat_dec,lon_dec,ttwa,pc4,lat_lon5,lat_lon4,lat_lon3
0,NI026041,"120B Malone Road, Belfast, County Antrim, BT9 5HT",BT9 5HT,pta,1,54.571000,-5.947278,NaN,BT9,"54.571,-5.94728","54.571,-5.9473","54.571,-5.947"
1,NI043471,"First Trust Centre,P.O.Box 123,9, Ann Street, ...",BT1 3AY,ro,1,54.599528,-5.922917,NaN,BT1,"54.59953,-5.92292","54.5995,-5.9229","54.6,-5.923"
2,NI036600,"At The Offices Of Tughan & Co, Marlborough Hou...",BT1 3GS,pta,2,99.999999,0.000000,NaN,BT1,"100.0,0.0","100.0,0.0","100.0,0.0"
3,NI004230,"City Quays 2, 8th Floor, 2 Clarendon Road, Bel...",BT1 3YD,ro,1,54.606750,-5.920500,NaN,BT1,"54.60675,-5.9205","54.6068,-5.9205","54.607,-5.921"
4,NI004467,"Donegall Square West, Belfast, County Antrim, ...",BT1 6JS,pta,1,54.596583,-5.931472,NaN,BT1,"54.59658,-5.93147","54.5966,-5.9315","54.597,-5.931"
...,...,...,...,...,...,...,...,...,...,...,...,...
562,10030036,"1 Floral Court Floral Street, London, WC2E 7FB",WC2E 7FB,ro,1,51.513417,-0.122750,NaN,WC2E,"51.51342,-0.12275","51.5134,-0.1228","51.513,-0.123"
563,05281259,"7th Floor 11 Strand, London, London, WC2R 5HR",WC2R 5HR,pta,1,51.507833,-0.126778,NaN,WC2R,"51.50783,-0.12678","51.5078,-0.1268","51.508,-0.127"
564,08849841,"Hill House 1 Little New Street, London, EC4A 3TA",EC4A 3TA,ro,1,51.515639,-0.106806,NaN,EC4A,"51.51564,-0.10681","51.5156,-0.1068","51.516,-0.107"
565,00976410,"Gresley House Ten Pound Walk, Doncaster, South...",DN5 4HX,ro,2,NaN,NaN,NaN,DN5,NaN,NaN,NaN


Latitude and Longitude (rounded to 5 decimal places) is null:


,registered_number,address_raw,extracted_postcode,address_case,address_lvl,lat_dec,lon_dec,ttwa,pc4,lat_lon5,lat_lon4,lat_lon3
0,04477951,"Bridge House, A430Ca9Llondon Bridge, London, S...",A43 0CA,ro,2,NaN,NaN,NaN,A43,NaN,NaN,NaN
1,05410859,"Speedwell Mill, Old Coach Road, Tansley, Matlo...",DE4 6FY,ro,2,NaN,NaN,NaN,DE4,NaN,NaN,NaN
2,09265490,"6 Office 6, Regus Ashford, Ashford, Kent, TN23...",TN23 8EZ,pta,2,NaN,NaN,NaN,TN23,NaN,NaN,NaN
3,02521977,"Sea Containers House, 18 Upper Ground, London,...",,ro,2,NaN,NaN,NaN,,NaN,NaN,NaN
4,00165727,"Shell Centre, York Road, London, SE1 7NA",,ro,2,NaN,NaN,NaN,,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
502,04802152,"PO Box 1390 Bradford, Bradford, West Yorkshire...",BD5 5FQ,pta,2,NaN,NaN,NaN,BD5,NaN,NaN,NaN
503,07530078,"C12 Marquis Court Marquisway, Team Valley, Gat...",NE1 0RU,ro,2,NaN,NaN,NaN,NE1,NaN,NaN,NaN
504,NI016601,"2 Washingford Row, Milltown, Dungannon, County...",BT71 3BG,ro,2,NaN,NaN,NaN,BT71,NaN,NaN,NaN
505,00976410,"Gresley House Ten Pound Walk, Doncaster, South...",DN5 4HX,ro,2,NaN,NaN,NaN,DN5,NaN,NaN,NaN


In [21]:
working_fixed_skinny = (
    working_fixed    
    .mutate(is_public = working_fixed.ticker_symbol.notnull())
    .select(
        # fame_fixed variables which we want to keep for the regression
        'registered_number', 'company_name', 'is_public', 'industry_codes', 'file_codes',
        'primary_uk_sic_2007_code', 'primary_uk_sic_2007_description',
    
        # new location variables
        'lat_dec', 'lon_dec', 'address_lvl', 'address_case', 'extracted_postcode', 'ttwa', 'pc4', 'lat_lon5'
    )
    .rename(
        pc8='extracted_postcode',
        sic6='primary_uk_sic_2007_code',
        sic6_desc='primary_uk_sic_2007_description'
    )
)

print(f"✅ Inserting columns into new '{new_table_name}' table.")
con.create_table(new_table_name, working_fixed_skinny, overwrite=True)

# Verify the final materialized table
final_table = con.table(new_table_name)
row_count = final_table.count().execute()
col_count = len(final_table.columns)

print(f"✅ Materialized '{new_table_name}' table.")
print(f"📊 Number of rows: {row_count:,}")
print(f"📊 Number of columns: {col_count}")
print(f"\nSample of {new_table_name}:")
display(final_table.sample(30 / row_count).execute())

✅ Inserting columns into new 'working_fixed' table.
✅ Materialized 'working_fixed' table.
📊 Number of rows: 152,379
📊 Number of columns: 15

Sample of working_fixed:


,registered_number,company_name,is_public,industry_codes,file_codes,sic6,sic6_desc,lat_dec,lon_dec,address_lvl,address_case,pc8,ttwa,pc4,lat_lon5
0,04352396,FIRSTPORT GROUP LIMITED,False,70,16_55,70100,Activities of head offices,50.755500,-1.671056,1,pta,BH25 5NR,E30000175,BH25,"50.7555,-1.67106"
1,02241211,WARWICK FABRICS (UK) LIMITED,False,"46,13","14_55,21_52",13300,Finishing of textiles,51.812201,-2.275917,2,pta,GL2 2BY,E30000209,GL2,"51.8122,-2.27592"
2,04947150,BMT FLEET TECHNOLOGY LIMITED,False,70,16_57,70229,Management consultancy activities (other than ...,51.241235,-0.562623,2,ro,GU1 1UN,E30000212,GU1,"51.24124,-0.56262"
3,00337663,BOC LIMITED,False,20,22_11,20110,Manufacture of industrial gases,51.238944,-0.613583,1,pta,GU21 6HT,E30000212,GU21,"51.23894,-0.61358"
4,03613589,C & P MEDICAL TRADING LIMITED,False,46,14_55,46460,Wholesale of pharmaceutical goods,51.358076,-2.129559,2,ro,SN12 6TP,E30000280,SN12,"51.35808,-2.12956"
5,02375184,THE OLD RECTORY (EWHURST) CO. LIMITED,False,87,19_22,87100,Residential nursing care activities,51.153917,-0.441556,1,pta,GU6 7RP,E30000212,GU6,"51.15392,-0.44156"
6,03734790,LEEDS SURVIVOR LED CRISIS SERVICES,False,86,17_50,86900,Other human health activities,53.798139,-1.466361,1,pta,LS15 7RW,E30000229,LS15,"53.79814,-1.46636"
7,14061918,CHELSEA GREEN MIDCO LIMITED,False,64,12_47,64303,Activities of venture and development capital ...,51.530383,-0.265655,2,ro,NW10 7NP,E30000234,NW10,"51.53038,-0.26565"
8,01248691,ACTCELL LIMITED,False,32,22_42,32120,Manufacture of jewellery and related articles,51.580917,-0.198667,1,pta,NW11 0PU,E30000234,NW11,"51.58092,-0.19867"
9,00414055,MECHANISED PROJECT MANAGEMENT,False,68,13_18,68209,Letting and operating of own or leased real es...,54.932083,-1.625333,1,pta,NE11 0BL,E30000245,NE11,"54.93208,-1.62533"


# 2. Geospatial group averages (tfp or similar)
**Groups (donut approach)**

1. Same building: address is equal, long/latitude is identical
2. 8-digit postcode is identical
3. 4-digit postcode is identical
4. NUTS-2 region

  5-8. Add 2-digit SIC codes

  9-12. Add 6-digit SIC codes

- Exclude the previous donut regions from each successive one
- Don’t spend too much time on this as this model is poorly identified anyway. Just do basic proof of concept / correlation and discuss the issues.
- **Separate regressions (baseline):** for each geographic grouping and SIC grouping one-by-one (12 models). Establishes baseline gross effect → expect to see weaker effects with each outside group.
- **Combined model:** final, preferred specification is combined model with all mutually exclusive rings (I can drop outer rings but not inner). Same with industry effect, can control. Including industries could distinguish knowledge spillovers (same supply chain, co-located) to localised same product market competition (same SIC code). Roughly correspond to Van Reenen, Bloom, Schankerman (2021).

**Data group assignment**

- roughly $O(nt\log [nt])=O(9m)$
1. Partition: by group string (as laid out above)
2. Aggregation: calculate average of remaining rows = sum -0 count
3. Assign to all filtered rows
4. Transformation: leave-one-out.

In [5]:
import ibis
from utils.f_0_dirs import get_data_dirs

table_name_fixed = "working_fixed"
table_name_yearly = "working_yearly"
table_name_peers = "working_yearly_peers"
column_peers = ["tfp", "gva1", "total_assets", "employees"]
spatial_groups = ["pc8", "pc4", "ttwa"]

dirs = get_data_dirs(segment="calculations")
con = ibis.duckdb.connect(str(dirs.db_path))
con.raw_sql("INSTALL spatial; LOAD spatial;")

table_fixed = con.table(table_name_fixed)
table_yearly = con.table(table_name_yearly)
table_joined = (
    table_yearly
    .distinct(on=['registered_number', 'year'])
    # Positive values only for the peer calculations. Generate filter condition that >0 for every column_peers
    .filter(ibis.and_(*[ibis._[c] > 0 for c in column_peers]))
    .left_join(
        table_fixed,
        "registered_number"
    )
)
columns_yearly_str = ", ".join(table_yearly.columns)

table_name_view = "spatial_panel_view"
con.create_view(table_name_view, table_joined, overwrite=True)

agg_sql_str = ""
calc_sql_str = ""
for i, g in enumerate(spatial_groups):

    for c in column_peers:
    
        prior_group = f"count_{c}_{spatial_groups[i-1]}" if i > 0 else '1'
        donut_str = '_d' if i > 0 else ''

        agg_sql_str += f"""
        SUM({c}) OVER (PARTITION BY {g}, year) AS sum_{c}_{g},
        COUNT({c}) OVER (PARTITION BY {g}, year) AS count_{c}_{g},
        """

        calc_sql_str += f"""
        (sum_{c}_{g} - {c}) / NULLIF(count_{c}_{g} - {prior_group}, 0) AS {c}_{g}{donut_str},
        """

# ==========================================
# 2. PARTITION, AGGREGATE, BROADCAST, TRANSFORM
# ==========================================
# We use 3 concentric rings. 
# Ring 1: extracted_postcode (Innermost)
# Ring 2: pc4 (Middle Donut)
# Ring 3: ttwa (Outer Donut)
window_query_body = f"""
WITH GroupAggregates AS (
    SELECT 
        {columns_yearly_str},
        {",\n".join(column_peers)},        
        {agg_sql_str}
    FROM 
        {table_name_view}
)
SELECT 
    registered_number,
    year,    
    {calc_sql_str}
    
FROM 
    GroupAggregates
"""

# 2. Write new table directly in the raw sql
create_table_ddl = (
    f"CREATE OR REPLACE TABLE {table_name_peers} AS ({window_query_body});"
)

# 3. Execute directly on DuckDB connection
con.raw_sql(create_table_ddl)

# 4. Bind the newly materialized physical table back to Ibis
table_peers = con.table(table_name_peers)
# View the schema to verify the 3 new mutually exclusive spatial donut columns
print(table_peers.schema())
# Print the head of the resulting table to verify the calculations
display(table_peers.sample(0.0001).execute())

ibis.Schema {
  registered_number    string
  year                 int64
  tfp_pc8              float64
  gva1_pc8             float64
  total_assets_pc8     float64
  employees_pc8        float64
  tfp_pc4_d            float64
  gva1_pc4_d           float64
  total_assets_pc4_d   float64
  employees_pc4_d      float64
  tfp_ttwa_d           float64
  gva1_ttwa_d          float64
  total_assets_ttwa_d  float64
  employees_ttwa_d     float64
}


,registered_number,year,tfp_pc8,gva1_pc8,total_assets_pc8,employees_pc8,tfp_pc4_d,gva1_pc4_d,total_assets_pc4_d,employees_pc4_d,tfp_ttwa_d,gva1_ttwa_d,total_assets_ttwa_d,employees_ttwa_d
0,08709411,2023,NaN,NaN,NaN,NaN,2.876256,6492.406734,2.393534e+04,119.924528,3.360557,13103.450266,7.330292e+04,230.955882
1,05192763,2007,2.988204,10703.874264,33574.441948,235.846154,5.328308,215375.013610,1.153271e+06,1397.681818,3.225666,75271.231128,7.390750e+05,813.583880
2,06192910,2011,3.619218,16947.957676,37661.315267,219.300000,3.576484,54498.566609,2.449917e+05,839.836207,3.206451,100949.741301,2.301179e+06,865.292993
3,SC092520,2021,NaN,NaN,NaN,NaN,3.015794,1313.959584,8.187752e+02,38.000000,2.952515,21748.145286,2.173160e+05,315.174619
4,04136968,2007,2.789178,33870.303119,130414.277870,740.529412,3.463084,16242.386507,7.070578e+04,259.529412,3.155468,15151.915282,6.285926e+04,326.303514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,04231521,2020,3.248076,22495.014966,50060.983209,731.333333,3.218896,8486.671533,1.865960e+04,307.911765,3.357810,13964.532408,5.731414e+04,262.135371
110,02214956,2019,NaN,NaN,NaN,NaN,2.777234,1972.002405,4.076715e+03,43.000000,3.201318,88278.181724,1.853400e+06,795.243337
111,05735653,2022,NaN,NaN,NaN,NaN,2.974986,4589.644387,8.374156e+03,105.933333,3.103540,36005.521877,1.649380e+05,808.130693
112,03618688,2014,3.059305,16140.884596,130300.858920,212.950000,3.857429,33398.546199,1.633221e+05,572.298507,4.931910,30288.534935,1.271811e+05,541.967480


In [6]:
con.raw_sql("CHECKPOINT;")
con.disconnect()